In [33]:
import psycopg2
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

user = os.getenv("user")
password = os.getenv("password")
port = os.getenv("port")
host = "localhost"
database= "crypto_analytics"

try:
    conexao = psycopg2.connect(
        host = host,
        database = database,
        user = user,
        password = password,
        port = port
    )
    print("Conectado com sucesso!")
except Exception as e:
    print(repr(e))

Conectado com sucesso!


In [34]:
query = "SELECT * FROM crypto_raw;"

with conexao.cursor() as cursor:
    cursor.execute(query)

    dados_raw = cursor.fetchall()

    colunas = [desc[0] for desc in cursor.description]

conexao.close()

df_raw = pd.DataFrame(dados_raw, columns=colunas)

display(df_raw)

,id,data_coleta,payload
0,2,2026-10-04 17:23:05.093532,"{'solana': {'brl': 630.26, 'eur': 107.45, 'usd..."
1,3,2026-10-04 17:24:08.552979,"{'solana': {'brl': 630.01, 'eur': 107.41, 'usd..."
2,4,2026-10-05 23:14:23.019484,"{'solana': {'brl': 601.74, 'eur': 107.35, 'usd..."
3,5,2026-10-06 00:00:19.286381,"{'solana': {'brl': 600.77, 'eur': 107.17, 'usd..."
4,6,2026-10-10 19:10:33.042844,"{'solana': {'brl': 549.75, 'eur': 98.39, 'usd'..."
5,7,2026-10-10 19:42:42.456010,"{'solana': {'brl': 550.82, 'eur': 98.59, 'usd'..."
6,8,2026-10-10 19:44:01.089508,"{'solana': {'brl': 550.68, 'eur': 98.56, 'usd'..."
7,9,2026-10-10 20:46:47.492044,"{'solana': {'brl': 549.03, 'eur': 98.27, 'usd'..."


In [35]:
registros = []

for _, linha in df_raw.iterrows():

    payload = linha["payload"]
    data_coleta = linha["data_coleta"]

    for coin, info in payload.items():
        registros.append({
            "coin": coin,
            "usd": info["usd"],
            "usd_market_cap": info["usd_market_cap"],
            "eur": info["eur"],
            "eur_market_cap": info["eur_market_cap"],
            "brl": info["brl"],
            "brl_market_cap": info["brl_market_cap"],
            "data_coleta": data_coleta
        })

df_silver = pd.DataFrame(registros)

display(df_silver)

,coin,usd,usd_market_cap,eur,eur_market_cap,brl,brl_market_cap,data_coleta
0,solana,120.960000,7.115170e+10,107.450000,6.320505e+10,630.260000,3.707288e+11,2026-10-04 17:23:05.093532
1,bitcoin,85473.000000,1.717318e+12,75927.000000,1.525518e+12,445349.000000,8.947913e+12,2026-10-04 17:23:05.093532
2,cardano,0.251861,9.444409e+09,0.223732,8.389601e+09,1.310000,4.920915e+10,2026-10-04 17:23:05.093532
3,ripples,0.001972,1.676277e+05,0.001752,1.489060e+05,0.010275,8.734071e+05,2026-10-04 17:23:05.093532
4,ethereum,2702.340000,3.299414e+11,2400.530000,2.930916e+11,14080.270000,1.719127e+12,2026-10-04 17:23:05.093532
5,binancecoin,790.010000,1.051859e+11,701.770000,9.343812e+10,4116.250000,5.480607e+11,2026-10-04 17:23:05.093532
6,solana,120.910000,7.113206e+10,107.410000,6.318760e+10,630.010000,3.706265e+11,2026-10-04 17:24:08.552979
7,bitcoin,85468.000000,1.717360e+12,75922.000000,1.525555e+12,445321.000000,8.948130e+12,2026-10-04 17:24:08.552979
8,cardano,0.251603,9.444898e+09,0.223503,8.390035e+09,1.310000,4.921170e+10,2026-10-04 17:24:08.552979
9,ripples,0.001972,1.676277e+05,0.001752,1.489060e+05,0.010275,8.734071e+05,2026-10-04 17:24:08.552979


In [36]:
tickers = {
    "Bitcoin":"BTC",
    "Ethereum":"ETH",
    "Solana":"SOL",
    "Ripples":"XRP",
    "Cardano":"ADA",
    "Binancecoin":"BNB"
}

df_silver["coin"] = df_silver["coin"].str.title()
df_silver["tickers"] = df_silver["coin"].map(tickers)

display(df_silver)

,coin,usd,usd_market_cap,eur,eur_market_cap,brl,brl_market_cap,data_coleta,tickers
0,Solana,120.960000,7.115170e+10,107.450000,6.320505e+10,630.260000,3.707288e+11,2026-10-04 17:23:05.093532,SOL
1,Bitcoin,85473.000000,1.717318e+12,75927.000000,1.525518e+12,445349.000000,8.947913e+12,2026-10-04 17:23:05.093532,BTC
2,Cardano,0.251861,9.444409e+09,0.223732,8.389601e+09,1.310000,4.920915e+10,2026-10-04 17:23:05.093532,ADA
3,Ripples,0.001972,1.676277e+05,0.001752,1.489060e+05,0.010275,8.734071e+05,2026-10-04 17:23:05.093532,XRP
4,Ethereum,2702.340000,3.299414e+11,2400.530000,2.930916e+11,14080.270000,1.719127e+12,2026-10-04 17:23:05.093532,ETH
5,Binancecoin,790.010000,1.051859e+11,701.770000,9.343812e+10,4116.250000,5.480607e+11,2026-10-04 17:23:05.093532,BNB
6,Solana,120.910000,7.113206e+10,107.410000,6.318760e+10,630.010000,3.706265e+11,2026-10-04 17:24:08.552979,SOL
7,Bitcoin,85468.000000,1.717360e+12,75922.000000,1.525555e+12,445321.000000,8.948130e+12,2026-10-04 17:24:08.552979,BTC
8,Cardano,0.251603,9.444898e+09,0.223503,8.390035e+09,1.310000,4.921170e+10,2026-10-04 17:24:08.552979,ADA
9,Ripples,0.001972,1.676277e+05,0.001752,1.489060e+05,0.010275,8.734071e+05,2026-10-04 17:24:08.552979,XRP


In [38]:
conexao = psycopg2.connect(
    host= host,
    database=  database,
    user= user,
    password= password,
    port= port
)

with conexao.cursor() as cursor:

    cursor.execute("""
    INSERT INTO crypto_silver (
        id_raw,
        coin,
        ticker,
        usd,
        usd_market_cap,
        eur,
        eur_market_cap,
        brl,
        brl_market_cap,
        data_coleta)
    VALUES (
        %s,%s,%s,%s,%s,%s,%s,%s,%s
    )
    """)

    conexao.commit()

conexao.close()

SyntaxError: ERRO:  erro de sintaxe em ou próximo a "%"
LINE 14:         %s,%s,%s,%s,%s,%s,%s,%s,%s
                 ^
